# Linked Hash Map

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Hash Tables, Linked List · **Difficulty/Frequency:** Uncommon (3/10)

> **Related:** [`10. LRU_Cache`](../10.%20LRU_Cache/10.%20LRU_Cache.ipynb) is this same structure in **access** order, with a capacity bound and an eviction rule. Build this one first and the LRU cache is three extra lines.

## Concepts

**What this problem is really testing:**
- That a hash map and an ordered sequence answer **different questions**, and how to compose them
- The difference between **insertion order** and **access order** — and that you must ask which one
- Doubly-linked-list splicing made safe by **sentinels**

**First-principles primer — what is each piece?**

- **Hash map.** Maps keys to values in O(1) average time by hashing the key to a slot. It knows *what*, not *when* — a bare hash table has no inherent order at all.
- **Doubly linked list.** Nodes chained with `prev` and `next`. It knows *order*, and it can splice a node out or in with a handful of pointer writes — **but only if you already hold that node**. Finding it by key would be O(n).
- **Linked hash map.** Both, sharing one set of node objects. The map finds the node; the list orders it. This is exactly Java's `LinkedHashMap`, and exactly what Python's `OrderedDict` is internally.
- **Sentinels.** Two permanent dummy nodes at the ends, never real entries. They guarantee every real node has neighbours on both sides, which deletes every "is this the first/last/only one?" branch.

**The design decision that carries everything:**

> **The map stores `key -> Node`, not `key -> value`.**

If it stored values, reordering an entry would mean *searching the list* for its node — O(n), and the whole point is gone. Because the map hands you the node directly, splicing it is a constant number of pointer writes.

**The question you must ask before writing code — insertion order or access order?**

Java's `LinkedHashMap` supports both, and they differ in exactly one place:

| | re-`put` an existing key | `get` an existing key |
|---|---|---|
| **insertion order** (Java default) | update the value, **position unchanged** | **position unchanged** |
| **access order** (`accessOrder=true`) | update the value, **move to the end** | **move to the end** |

Access order is the LRU building block — "most recently touched at the end" is precisely the recency list an LRU cache evicts from. Insertion order is what you want for a configuration map, a JSON object, or anything where the caller's declaration sequence is meaningful.

Both are defensible; **silently picking one is not**. Say which you are building and why.

**Simple worked example.** `put(a,1)`, `put(b,2)`, `put(c,3)`, then `put(a,99)`, then `get(b)`:

| after | insertion order | access order |
|---|---|---|
| the three puts | `a, b, c` | `a, b, c` |
| `put(a, 99)` | `a, b, c` — value changed, position kept | `b, c, a` — moved to the end |
| `get(b)` | `a, b, c` — reads change nothing | `c, a, b` — moved to the end |

## Problem Statement

Implement a `LinkedHashMap`: a hash map whose iteration order is well-defined.

| Method | Behaviour |
|---|---|
| `put(key, value)` | Insert or update; returns the previous value, or `None` |
| `get(key)` | The value, or `None` |
| `remove(key)` | Remove and return the value, or `None` |
| `__contains__`, `__len__` | Membership and size |
| `__iter__`, `items()`, `values()` | Iterate **in order** |

All of `put` / `get` / `remove` / `in` must be **O(1)** average.

**Example**

```python
m = LinkedHashMap()
m.put("a", 1); m.put("b", 2); m.put("c", 3)
list(m)              # -> ["a", "b", "c"]
m.put("a", 99)       # update - insertion order keeps a first
list(m)              # -> ["a", "b", "c"]
m.remove("b")
list(m)              # -> ["a", "c"]
```

### Approach 1 — Naive (a list of key/value pairs)

**Idea:** keep pairs in a plain list. Order comes free; everything else is a scan.

This is the honest baseline: it satisfies the *ordering* requirement perfectly and fails the *complexity* one completely.

**Time complexity:** **O(n)** for `get`, `put` and `remove`.

**Space complexity:** O(n).

In [ ]:
from typing import Any, Dict, Iterator, List, Optional, Tuple


class NaiveLinkedHashMap:
    """Baseline: order is free, but every operation is a linear scan."""

    def __init__(self) -> None:
        self.pairs: List[List[Any]] = []                 # [key, value] in insertion order

    def put(self, key: Any, value: Any) -> Any:
        for pair in self.pairs:                          # O(n) scan
            if pair[0] == key:
                old, pair[1] = pair[1], value
                return old                               # update in place: position kept
        self.pairs.append([key, value])
        return None

    def get(self, key: Any, default: Any = None) -> Any:
        for k, v in self.pairs:                          # O(n)
            if k == key:
                return v
        return default

    def remove(self, key: Any) -> Any:
        for i, (k, v) in enumerate(self.pairs):
            if k == key:
                self.pairs.pop(i)                        # O(n) scan + O(n) shift
                return v
        return None

    def __contains__(self, key: Any) -> bool:
        return any(k == key for k, _ in self.pairs)

    def __len__(self) -> int:
        return len(self.pairs)

    def __iter__(self) -> Iterator[Any]:
        return iter([k for k, _ in self.pairs])

    def items(self) -> Iterator[Tuple[Any, Any]]:
        return iter([(k, v) for k, v in self.pairs])

### Approach 2 — Optimal (hash map of nodes + doubly linked list, both orderings)

**Idea:** the dict finds the node in O(1); the list holds the order. One `access_order` flag decides whether a touch relocates the node.

**Three details worth defending:**

- **Sentinels make splicing branch-free.** With permanent `head` and `tail` dummies, `_unlink` is unconditionally two assignments and `_append_to_tail` four. Without them you need cases for empty / single / first / last — four branches in the two functions most likely to be written wrong under pressure.
- **`_unlink` clears the node's own pointers afterwards.** `node.prev = node.next = None` is not required for correctness of the list, but it turns "accidentally reused a stale node" from a silent corruption into an immediate `AttributeError`, and it lets the garbage collector reclaim neighbours the node would otherwise pin.
- **The node stores its own key.** Redundant here, but it is what lets `popitem()` (and, in the LRU next door, eviction) delete the matching *map* entry when all you hold is a *node*.

**Why not just use a `dict`?** Python 3.7+ dicts *do* preserve insertion order — but they cannot **reorder**. There is no "move this key to the end" operation, which is exactly what access-order mode and every LRU cache need. That, plus explicit control of the ordering policy, is what the linked list buys.

**Time complexity:** **O(1)** average for `put`, `get`, `remove`, `in`; O(n) to iterate.

**Space complexity:** O(n) — the map and the list hold *the same* nodes.

In [ ]:
class _Node:
    __slots__ = ("key", "value", "prev", "next")

    def __init__(self, key: Any = None, value: Any = None) -> None:
        self.key = key                    # the node carries its own key, for popitem/eviction
        self.value = value
        self.prev: Optional["_Node"] = None
        self.next: Optional["_Node"] = None


class LinkedHashMap:
    """Hash map + doubly linked list. access_order=False gives Java's default behaviour."""

    def __init__(self, access_order: bool = False) -> None:
        self.access_order = access_order
        self._map: Dict[Any, _Node] = {}          # key -> the SAME node that is in the list
        self._head = _Node()                      # sentinel: oldest end
        self._tail = _Node()                      # sentinel: newest end
        self._head.next = self._tail
        self._tail.prev = self._head

    # ---- list primitives: no branches, thanks to the sentinels ----------
    def _unlink(self, node: _Node) -> None:
        node.prev.next = node.next                # both neighbours always exist
        node.next.prev = node.prev
        node.prev = node.next = None              # a stale reuse now fails loudly, not silently

    def _append_to_tail(self, node: _Node) -> None:
        last = self._tail.prev
        last.next = node
        node.prev = last
        node.next = self._tail
        self._tail.prev = node

    def _touch(self, node: _Node) -> None:
        """Move to the newest end - but ONLY in access-order mode."""
        if self.access_order:
            self._unlink(node)
            self._append_to_tail(node)

    # ---- public API ------------------------------------------------------
    def put(self, key: Any, value: Any) -> Any:
        node = self._map.get(key)
        if node is not None:
            old = node.value
            node.value = value
            self._touch(node)                     # insertion order: position UNCHANGED
            return old                            # returning the previous value, like Java's put
        node = _Node(key, value)
        self._map[key] = node
        self._append_to_tail(node)                # a NEW key always goes to the end
        return None

    def get(self, key: Any, default: Any = None) -> Any:
        node = self._map.get(key)
        if node is None:
            return default
        self._touch(node)                         # in access order, a read counts as a use
        return node.value

    def remove(self, key: Any) -> Any:
        node = self._map.pop(key, None)
        if node is None:
            return None
        self._unlink(node)                        # remove from BOTH structures, or they drift
        return node.value

    def popitem(self, last: bool = True) -> Tuple[Any, Any]:
        """Remove and return the newest (last=True) or oldest (last=False) entry."""
        if not self._map:
            raise KeyError("popitem from an empty LinkedHashMap")
        node = self._tail.prev if last else self._head.next
        del self._map[node.key]                   # THIS is why the node stores its own key
        self._unlink(node)
        return node.key, node.value

    def __contains__(self, key: Any) -> bool:
        return key in self._map

    def __len__(self) -> int:
        return len(self._map)

    def __iter__(self) -> Iterator[Any]:
        node = self._head.next
        while node is not self._tail:
            nxt = node.next                       # read next FIRST: the caller may delete this key
            yield node.key
            node = nxt

    def items(self) -> Iterator[Tuple[Any, Any]]:
        node = self._head.next
        while node is not self._tail:
            nxt = node.next
            yield node.key, node.value
            node = nxt

    def values(self) -> Iterator[Any]:
        for _, v in self.items():
            yield v

    def __eq__(self, other: object) -> bool:
        """Equal iff the same key/value pairs appear in the same ORDER."""
        if not isinstance(other, LinkedHashMap):
            return NotImplemented
        return list(self.items()) == list(other.items())

    def __repr__(self) -> str:
        inner = ", ".join(f"{k!r}: {v!r}" for k, v in self.items())
        return f"LinkedHashMap({{{inner}}}, access_order={self.access_order})"

### Follow-up — an LRU cache, in three extra lines

**Idea:** this is the payoff for building the general structure. An LRU cache **is** a linked hash map in access order, plus a capacity bound.

- **Access order** already keeps the most recently touched entry at the tail, which means the **least** recently used is at the head.
- The only new logic: after inserting, if `len > capacity`, `popitem(last=False)`.

That is the entire difference. Worth saying out loud, because recognising that one problem is a special case of another is exactly what an interviewer is listening for.

**Time complexity:** O(1) per operation.

**Space complexity:** O(capacity).

In [ ]:
class LRUCache(LinkedHashMap):
    """A LinkedHashMap in access order, with a capacity bound. That is all an LRU cache is."""

    def __init__(self, capacity: int) -> None:
        super().__init__(access_order=True)       # <- reads and writes both refresh recency
        self.capacity = capacity

    def put(self, key: Any, value: Any) -> Any:
        old = super().put(key, value)
        if self.capacity <= 0:
            self._map.clear()                     # a zero-capacity cache stores nothing
            self._head.next, self._tail.prev = self._tail, self._head
        elif len(self) > self.capacity:
            self.popitem(last=False)              # <- evict the OLDEST end. One line.
        return old

## Verification

Check both ordering modes, the `remove`/`popitem` bookkeeping, and — most importantly — that the map and the list never disagree.

In [ ]:
import random

# --- Insertion order (Java's default): updates do NOT move the entry ---
m = LinkedHashMap()
assert m.put("a", 1) is None, "put returns the PREVIOUS value, None for a new key"
m.put("b", 2)
m.put("c", 3)
assert list(m) == ["a", "b", "c"]
assert m.put("a", 99) == 1, "put returns the value it replaced"
assert list(m) == ["a", "b", "c"], "insertion order: an update must NOT move the entry"
assert m.get("a") == 99
assert list(m) == ["a", "b", "c"], "insertion order: a read must NOT move the entry"
assert list(m.items()) == [("a", 99), ("b", 2), ("c", 3)]
assert list(m.values()) == [99, 2, 3]

# --- Access order: both get and put move the entry to the end ---
a = LinkedHashMap(access_order=True)
a.put("a", 1); a.put("b", 2); a.put("c", 3)
assert list(a) == ["a", "b", "c"]
a.put("a", 99)
assert list(a) == ["b", "c", "a"], "access order: an update moves the entry to the end"
a.get("b")
assert list(a) == ["c", "a", "b"], "access order: a read moves the entry to the end"
assert a.get("missing") is None
assert list(a) == ["c", "a", "b"], "a MISS must not reorder anything"

# --- remove, membership, length ---
m = LinkedHashMap()
for k, v in [("x", 1), ("y", 2), ("z", 3)]:
    m.put(k, v)
assert len(m) == 3 and "y" in m
assert m.remove("y") == 2
assert list(m) == ["x", "z"] and len(m) == 2 and "y" not in m
assert m.remove("y") is None, "removing an absent key returns None, it does not raise"
assert m.get("y") is None

# --- popitem from either end ---
m = LinkedHashMap()
for k in "abcd":
    m.put(k, k.upper())
assert m.popitem() == ("d", "D"), "last=True pops the newest"
assert m.popitem(last=False) == ("a", "A"), "last=False pops the oldest"
assert list(m) == ["b", "c"]
empty = LinkedHashMap()
try:
    empty.popitem()
except KeyError:
    pass
else:
    raise AssertionError("popitem on an empty map must raise KeyError")

# --- Edge cases ---
e = LinkedHashMap()
assert list(e) == [] and len(e) == 0 and e.get("k") is None and "k" not in e
one = LinkedHashMap(); one.put("solo", 1)
assert list(one) == ["solo"]
one.remove("solo")
assert list(one) == [] and len(one) == 0
one.put("again", 2)                                # reusing a drained map must still work
assert list(one) == ["again"]

# Falsy and None values are stored, not confused with absence
f = LinkedHashMap()
for k, v in [("zero", 0), ("empty", ""), ("false", False), ("none", None)]:
    f.put(k, v)
assert f.get("zero") == 0 and f.get("false") is False and f.get("empty") == ""
assert f.get("none") is None and "none" in f, "a stored None is present, unlike a missing key"
assert len(f) == 4

# --- Equality compares ORDER as well as content ---
p, q = LinkedHashMap(), LinkedHashMap()
p.put("a", 1); p.put("b", 2)
q.put("a", 1); q.put("b", 2)
assert p == q
r = LinkedHashMap()
r.put("b", 2); r.put("a", 1)                       # same pairs, different order
assert p != r, "a linked hash map's identity includes its order"

# --- The map and the list must never disagree ---
def check_consistency(m: LinkedHashMap) -> None:
    forward, node = [], m._head.next
    while node is not m._tail:
        forward.append(node)
        node = node.next
    backward, node = [], m._tail.prev
    while node is not m._head:
        backward.append(node)
        node = node.prev
    assert forward == backward[::-1], "prev/next disagree - the list is corrupt"
    assert len(forward) == len(m._map), "the list and the map hold different counts"
    for n in forward:
        assert m._map[n.key] is n, "the map must point at the very node in the list"


# --- Randomised: both implementations must agree, in both modes ---
random.seed(31)
for access in (False, True):
    fast = LinkedHashMap(access_order=access)
    naive = NaiveLinkedHashMap()                   # naive only models INSERTION order
    for step in range(4000):
        k = f"k{random.randrange(15)}"
        r = random.random()
        if r < 0.45:
            v = random.randrange(100)
            assert fast.put(k, v) == naive.put(k, v), step
        elif r < 0.8:
            got = fast.get(k)
            if not access:
                assert got == naive.get(k), step
        else:
            assert fast.remove(k) == naive.remove(k), step
        if not access:
            assert list(fast.items()) == list(naive.items()), f"order diverged at {step}"
        assert len(fast) == len(naive)
        if step % 200 == 0:
            check_consistency(fast)
    check_consistency(fast)

# --- The LRU cache follow-up ---
c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
assert c.get(1) == 1                               # touching 1 protects it
c.put(3, 3)                                        # over capacity -> evict the oldest, which is 2
assert c.get(2) is None and list(c) == [1, 3]
c.put(4, 4)                                        # -> evicts 1
assert c.get(1) is None and c.get(3) == 3 and c.get(4) == 4
assert len(c) == 2
check_consistency(c)

zero = LRUCache(0)
zero.put("a", 1)
assert len(zero) == 0 and zero.get("a") is None

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **`put` returning the previous value.** Implemented above, because it is what Java's `Map.put` contract specifies and it costs one line. The subtlety worth naming: `None` is ambiguous — it means both "no previous value" and "the previous value was `None`". Java has the same wart. If it matters, return a sentinel or a `(had_previous, value)` pair — the same trap as the `null` sentinel in [`7. Deep_Key_Search_Nested_JSON`](../7.%20Deep_Key_Search_Nested_JSON/7.%20Deep_Key_Search_Nested_JSON.ipynb).
- **`__eq__` including order.** Also implemented. This is a genuine design question, not a formality: Java's `LinkedHashMap.equals` **ignores** order (it inherits `AbstractMap.equals`), so two linked hash maps with the same pairs in different orders are *equal* in Java. Ordering it, as here, is defensible for a structure whose whole purpose is order — but it breaks the substitutability of `LinkedHashMap` for `Map`. State which you chose and why.
- **Worst-case complexity.** Everything above is O(1) *average*. Worst case, every key collides and lookup degrades to O(n). CPython's `dict` uses **open addressing with probing** (not bucket chains) and resizes at about two-thirds load, so this needs adversarially chosen keys to trigger — which is a real attack (hash-flooding), and why Python randomises string hashes per process by default. The linked-list half is genuinely O(1) worst case; only the hashing half can degrade.
- **Iterating while mutating.** The generators above read `node.next` *before* yielding, so deleting the current key mid-iteration is safe. Deleting the *next* one is not, and neither is inserting. Java throws `ConcurrentModificationException` for exactly this. If you need full safety, iterate a snapshot — the same conclusion as [`9. Broadcast_Message_Bus`](../9.%20Broadcast_Message_Bus/9.%20Broadcast_Message_Bus.ipynb).
- **Thread safety.** A single lock around every method is correct and cheap, since each operation is already O(1). Note the asymmetry: in **insertion-order** mode `get` is a genuine read, so a readers–writer lock lets reads run in parallel. In **access-order** mode `get` *mutates the list*, so every operation is a writer and the reader-writer split buys nothing — the same surprising property as the LRU cache.
- **Why not just use a `dict`?** Python 3.7+ dicts preserve insertion order, so for that mode alone a plain dict is enough — and `popitem(last=False)` even exists on `OrderedDict`. What a dict cannot do is **reorder**: there is no "move this key to the end". That single missing operation is why access-order mode, and every LRU cache, needs the explicit linked list.

## Empirical complexity check

Compare the **list-of-pairs** map (Approach 1, O(n) per operation) with the **hash map + linked list** (Approach 2, O(1)), running a fixed number of mixed operations against a map whose **size** doubles.

| Growth when the entry count doubles | What it means |
|---|---|
| ~2x | linear — every operation scans the whole map |
| ~1x | constant — a hash lookup plus a fixed number of pointer writes |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random

OPS = 20000


def make_workload(n):
    rng = random.Random(37)
    keys = [f"k{i}" for i in range(n)]
    ops = [(rng.random(), rng.choice(keys)) for _ in range(OPS)]
    return (keys, ops)


def _drive(m, keys, ops):
    for k in keys:
        m.put(k, 0)                       # prefill, so every op works against a full map
    for r, k in ops:
        if r < 0.4:
            m.put(k, 1)
        elif r < 0.9:
            m.get(k)
        else:
            m.remove(k)
            m.put(k, 2)                   # put it back, so the size stays steady


def run_naive(keys, ops):
    _drive(NaiveLinkedHashMap(), keys, ops)       # O(n) scan per operation


def run_optimal(keys, ops):
    _drive(LinkedHashMap(), keys, ops)            # O(1): hash lookup + pointer writes


benchmark(
    {"Approach 1 - list of pairs O(n)": run_naive,
     "Approach 2 - map + linked list O(1)": run_optimal},
    make_workload,
    sizes=[250, 500, 1000, 2000],
    repeats=1,
)

## Patterns learned

- **Compose two structures when one cannot answer both questions.** Hash map for *identity*, linked list for *order*, sharing one set of nodes. The same pairing behind [`10. LRU_Cache`](../10.%20LRU_Cache/10.%20LRU_Cache.ipynb), LFU caches, timer wheels and schedulers.
- **Store references, not copies.** The map holds the very node the list holds. That is what makes the cross-structure operation O(1) instead of a search — and why the space is O(n), not O(2n).
- **Ask which ordering the caller means.** Insertion order and access order differ in exactly one line, and produce completely different structures. Picking silently is the mistake; naming the choice is the answer.
- **Sentinels turn branches into straight-line code.** Two dummy nodes remove every empty/single/first/last case from the two functions most likely to be written wrong.
- **A generalisation is worth more than a special case.** Building the linked hash map first makes the LRU cache a three-line subclass. Recognising that one problem *is* another with a constraint added is what separates a strong answer from a correct one.
- **Read `next` before you yield.** One line that makes deleting the current key mid-iteration safe.
- **State the invariant and test it directly.** *"The list holds exactly the map's keys, in order, and the map points at the very nodes in the list."* `check_consistency` turns that sentence into a test that catches every pointer bug.